In [1]:
from pathlib import Path
import sys
from datetime import datetime

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "ruleset_comparison_zero"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")
EXPERIMENT_ID = f"exp_ruleset_comparison_zero_{RUN_TS}"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("EXPERIMENT_ID:", EXPERIMENT_ID)

PROJECT_ROOT: /home/harielpadillasanchez/Documentos/TT/TT2
DATA_DIR: /home/harielpadillasanchez/Documentos/TT/TT2/data
OUTPUT_DIR: /home/harielpadillasanchez/Documentos/TT/TT2/outputs/ruleset_comparison_zero
EXPERIMENT_ID: exp_ruleset_comparison_zero_20260502_235933


In [2]:
import json
import pandas as pd
import numpy as np

from configs.models import MODELS
from configs.rules import RULESETS
from src.experiment.runner import ExperimentRunner
from src.evaluation.metrics import evaluate_dataframe, summarize_metrics

/home/harielpadillasanchez/Documentos/TT/TT2/.venv-bloom/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
SAMPLE_PATH = DATA_DIR / "Muestra_csv.csv"

df_sample = pd.read_csv(SAMPLE_PATH)

print("Shape:", df_sample.shape)
print("Columnas:", list(df_sample.columns))
display(df_sample.head(3))

Shape: (20, 17)
Columnas: ['id', 'idFinal', 'grupo', 'motivo', 'Segmento', 'Propuesta', 'idcod', 'atr0', 'atr1', 'atr2', 'atr3', 'atr4', 'atr5', 'atr6', 'atr7', 'atr8', 'lex']


,id,idFinal,grupo,motivo,Segmento,Propuesta,idcod,atr0,atr1,atr2,atr3,atr4,atr5,atr6,atr7,atr8,lex
0,2088,1872_LibroBAC.pdf,A_cortos,"Muy corto, sustitución léxica clara.",La comunidad humana más antigua ha sido denomi...,La comunidad humana más antigua se llamó tribu.,Vivian,10,1,4,5.0,NaN,NaN,NaN,NaN,NaN,1
1,2976,881_LibroNEFE_Sincopyright.pdf,A_cortos,Corto con dinero/cantidad.,Prueba a poner en un sobre o frasco $1 por día...,Pon en un sobre un dólar diario más las moneda...,Vivian,14,5,1,6.0,NaN,NaN,NaN,NaN,NaN,1
2,3430,2692_LibroUide_Sincopyright.pdf,A_cortos,Frase conceptual simple de negocios.,Uno de los problemas más importantes en los ne...,Asignar precios a los productos es un problema...,Vivian,5,6,19,1.0,NaN,NaN,NaN,NaN,NaN,1


In [4]:
META_COLS = ["idFinal", "grupo", "motivo", "lex"]

required_cols = ["id", "Segmento", "Propuesta"]
missing = [c for c in required_cols if c not in df_sample.columns]
if missing:
    raise ValueError(f"Faltan columnas requeridas en la muestra: {missing}")

df_refine = df_sample.copy()
df_refine = df_refine.rename(
    columns={
        "id": "sample_id",
        "Segmento": "source_text",
        "Propuesta": "reference_text",
    }
)

keep_cols = ["sample_id", "source_text", "reference_text"] + [c for c in META_COLS if c in df_refine.columns]
df_refine = df_refine[keep_cols].copy()

print("Shape refinamiento:", df_refine.shape)
display(df_refine.head(5))

Shape refinamiento: (20, 7)


,sample_id,source_text,reference_text,idFinal,grupo,motivo,lex
0,2088,La comunidad humana más antigua ha sido denomi...,La comunidad humana más antigua se llamó tribu.,1872_LibroBAC.pdf,A_cortos,"Muy corto, sustitución léxica clara.",1
1,2976,Prueba a poner en un sobre o frasco $1 por día...,Pon en un sobre un dólar diario más las moneda...,881_LibroNEFE_Sincopyright.pdf,A_cortos,Corto con dinero/cantidad.,1
2,3430,Uno de los problemas más importantes en los ne...,Asignar precios a los productos es un problema...,2692_LibroUide_Sincopyright.pdf,A_cortos,Frase conceptual simple de negocios.,1
3,3679,"El resultado será 190, o sea, el IVA del kilo ...","El resultado será ciento noventa. Es decir, el...",829_LibroUAC_Sincopyright.pdf,A_cortos,Corto con número e IVA.,1
4,3145,"Por cada $100 facturados, la compañía gasta $2...",La empresa gasta dos dólares con veinte centav...,1045_LibroUide_Sincopyright.pdf,A_cortos,Corto con proporción financiera.,1


In [5]:
PROMPT_TYPE = "zero-shot"
FEW_SHOT_EXAMPLES = None
FEW_SHOT_EXAMPLE_IDS = []

print("PROMPT_TYPE:", PROMPT_TYPE)
print("FEW_SHOT_EXAMPLES:", FEW_SHOT_EXAMPLES)
print("FEW_SHOT_EXAMPLE_IDS:", FEW_SHOT_EXAMPLE_IDS)

PROMPT_TYPE: zero-shot
FEW_SHOT_EXAMPLES: None
FEW_SHOT_EXAMPLE_IDS: []


In [6]:
FINALIST_CONFIGS = [
    # HARIEL
    {
        "owner": "hariel",
        "model_key": "llama3",
        "config_label": "hariel_llama3_cfg_1",
        "temperature": 0.2,
        "top_p": 0.85,
        "repetition_penalty": 1.05,
        "max_new_tokens": 256,
    },
    {
        "owner": "hariel",
        "model_key": "llama3",
        "config_label": "hariel_llama3_cfg_2",
        "temperature": 0.2,
        "top_p": 0.90,
        "repetition_penalty": 1.10,
        "max_new_tokens": 256,
    },
    {
        "owner": "hariel",
        "model_key": "llama3",
        "config_label": "hariel_llama3_cfg_3",
        "temperature": 0.3,
        "top_p": 0.90,
        "repetition_penalty": 1.15,
        "max_new_tokens": 256,
    },
    {
        "owner": "hariel",
        "model_key": "mistral",
        "config_label": "hariel_mistral_cfg_1",
        "temperature": 0.3,
        "top_p": 0.90,
        "repetition_penalty": 1.10,
        "max_new_tokens": 256,
    },
    {
        "owner": "hariel",
        "model_key": "mistral",
        "config_label": "hariel_mistral_cfg_2",
        "temperature": 0.1,
        "top_p": 0.85,
        "repetition_penalty": 1.10,
        "max_new_tokens": 256,
    },
    {
        "owner": "hariel",
        "model_key": "mistral",
        "config_label": "hariel_mistral_cfg_3",
        "temperature": 0.2,
        "top_p": 0.85,
        "repetition_penalty": 1.10,
        "max_new_tokens": 256,
    },

    # NANCY
    {
        "owner": "nancy",
        "model_key": "llama3",
        "config_label": "nancy_llama3_cfg_1",
        "temperature": 0.3,
        "top_p": 0.90,
        "repetition_penalty": 1.15,
        "max_new_tokens": 256,
        "do_sample": True,
        "no_repeat_ngram_size": 4,
    },
    {
        "owner": "nancy",
        "model_key": "llama3",
        "config_label": "nancy_llama3_cfg_2",
        "temperature": 0.7,
        "top_p": 0.90,
        "repetition_penalty": 1.10,
        "max_new_tokens": 512,
        "do_sample": True,
        "no_repeat_ngram_size": 4,
    },
    {
        "owner": "nancy",
        "model_key": "mistral",
        "config_label": "nancy_mistral_cfg_1",
        "temperature": 0.7,
        "top_p": 0.90,
        "repetition_penalty": 1.10,
        "max_new_tokens": 512,
        "do_sample": True,
        "no_repeat_ngram_size": 4,
    },
    {
        "owner": "nancy",
        "model_key": "mistral",
        "config_label": "nancy_mistral_cfg_2",
        "temperature": 0.3,
        "top_p": 0.90,
        "repetition_penalty": 1.15,
        "max_new_tokens": 400,
        "do_sample": True,
        "no_repeat_ngram_size": 4,
    },
]

ACTIVE_RULESETS = ["R0", "R1", "R2", "R3", "R4"]

print("Prompt type fijo:", PROMPT_TYPE)
print("N configuraciones finalistas unificadas:", len(FINALIST_CONFIGS))
print("Rulesets:", ACTIVE_RULESETS)
display(pd.DataFrame(FINALIST_CONFIGS))

Prompt type fijo: zero-shot
N configuraciones finalistas unificadas: 10
Rulesets: ['R0', 'R1', 'R2', 'R3', 'R4']


,owner,model_key,config_label,temperature,top_p,repetition_penalty,max_new_tokens,do_sample,no_repeat_ngram_size
0,hariel,llama3,hariel_llama3_cfg_1,0.2,0.85,1.05,256,NaN,NaN
1,hariel,llama3,hariel_llama3_cfg_2,0.2,0.90,1.10,256,NaN,NaN
2,hariel,llama3,hariel_llama3_cfg_3,0.3,0.90,1.15,256,NaN,NaN
3,hariel,mistral,hariel_mistral_cfg_1,0.3,0.90,1.10,256,NaN,NaN
4,hariel,mistral,hariel_mistral_cfg_2,0.1,0.85,1.10,256,NaN,NaN
5,hariel,mistral,hariel_mistral_cfg_3,0.2,0.85,1.10,256,NaN,NaN
6,nancy,llama3,nancy_llama3_cfg_1,0.3,0.90,1.15,256,True,4.0
7,nancy,llama3,nancy_llama3_cfg_2,0.7,0.90,1.10,512,True,4.0
8,nancy,mistral,nancy_mistral_cfg_1,0.7,0.90,1.10,512,True,4.0
9,nancy,mistral,nancy_mistral_cfg_2,0.3,0.90,1.15,400,True,4.0


In [7]:
for cfg in FINALIST_CONFIGS:
    if cfg["model_key"] not in MODELS:
        raise ValueError(f"Modelo no definido en MODELS: {cfg['model_key']}")

for ruleset in ACTIVE_RULESETS:
    if ruleset not in RULESETS:
        raise ValueError(f"Ruleset no definido en RULESETS: {ruleset}")

if PROMPT_TYPE not in ["zero-shot", "few-shot"]:
    raise ValueError(f"Técnica no soportada: {PROMPT_TYPE}")

print("Configuración validada correctamente.")

Configuración validada correctamente.


In [8]:
runner = ExperimentRunner(
    experiment_id=EXPERIMENT_ID,
    log_dir=str(PROJECT_ROOT / "outputs" / "logs")
)

print("Runner inicializado:", runner.experiment_id)

Runner inicializado: exp_ruleset_comparison_zero_20260502_235933


In [9]:
test_row = df_refine.iloc[0]
test_cfg = FINALIST_CONFIGS[0]

generation_config_test = {
    "temperature": test_cfg["temperature"],
    "top_p": test_cfg["top_p"],
    "repetition_penalty": test_cfg["repetition_penalty"],
    "max_new_tokens": test_cfg["max_new_tokens"],
}

if "do_sample" in test_cfg:
    generation_config_test["do_sample"] = test_cfg["do_sample"]
if "no_repeat_ngram_size" in test_cfg:
    generation_config_test["no_repeat_ngram_size"] = test_cfg["no_repeat_ngram_size"]

test_record = runner.run_one(
    dataset_name="muestra_20_ruleset_comparison_zero",
    model_key=test_cfg["model_key"],
    prompt_type=PROMPT_TYPE,
    ruleset=ACTIVE_RULESETS[0],
    source_text=test_row["source_text"],
    reference_text=test_row["reference_text"],
    sample_id=str(test_row["sample_id"]),
    fold_id=None,
    split_name="ruleset_comparison_zero",
    few_shot_examples=None,
    few_shot_example_ids=None,
    generation_config=generation_config_test,
)

test_record.to_dict()

{'experiment_id': 'exp_ruleset_comparison_zero_20260502_235933',
 'run_id': '1901c694-6b3d-4c2f-ba05-fc6772951f3c',
 'timestamp': '2026-05-02T23:59:38.855427',
 'dataset_name': 'muestra_20_ruleset_comparison_zero',
 'fold_id': None,
 'split_name': 'ruleset_comparison_zero',
 'model_key': 'llama3',
 'model_id': 'meta-llama/Meta-Llama-3-8B-Instruct',
 'backend': 'ollama',
 'prompt_type': 'zero-shot',
 'ruleset': 'R0',
 'few_shot_example_ids': [],
 'generation_config': {'temperature': 0.2,
  'top_p': 0.85,
  'repetition_penalty': 1.05,
  'max_new_tokens': 256},
 'sample_id': '2088',
 'source_text': 'La comunidad humana más antigua ha sido denominada horda primitiva.',
 'reference_text': 'La comunidad humana más antigua se llamó tribu.',
 'generated_text': 'La comunidad humana más antigua se llama horda primitiva.',
 'prompt_text': 'Reescribe en español el siguiente texto con lenguaje más claro y sencillo.\nConserva el significado original y no inventes información.\n\nDevuelve solo la ver

In [10]:
all_records = []

total_runs = len(FINALIST_CONFIGS) * len(ACTIVE_RULESETS) * len(df_refine)
run_counter = 0

for cfg in FINALIST_CONFIGS:
    for ruleset in ACTIVE_RULESETS:
        for _, row in df_refine.iterrows():
            run_counter += 1

            generation_config = {
                "temperature": cfg["temperature"],
                "top_p": cfg["top_p"],
                "repetition_penalty": cfg["repetition_penalty"],
                "max_new_tokens": cfg["max_new_tokens"],
            }

            if "do_sample" in cfg:
                generation_config["do_sample"] = cfg["do_sample"]
            if "no_repeat_ngram_size" in cfg:
                generation_config["no_repeat_ngram_size"] = cfg["no_repeat_ngram_size"]

            print(
                f"[{run_counter}/{total_runs}] "
                f"owner={cfg['owner']} | "
                f"model={cfg['model_key']} | "
                f"cfg={cfg['config_label']} | "
                f"ruleset={ruleset} | "
                f"sample_id={row['sample_id']}"
            )

            record = runner.run_one(
                dataset_name="muestra_20_ruleset_comparison_zero",
                model_key=cfg["model_key"],
                prompt_type=PROMPT_TYPE,
                ruleset=ruleset,
                source_text=row["source_text"],
                reference_text=row["reference_text"],
                sample_id=str(row["sample_id"]),
                fold_id=None,
                split_name="ruleset_comparison_zero",
                few_shot_examples=None,
                few_shot_example_ids=None,
                generation_config=generation_config,
            )

            record_dict = record.to_dict()
            record_dict["owner"] = cfg["owner"]
            record_dict["config_label"] = cfg["config_label"]

            for extra_col in [
                "temperature", "top_p", "repetition_penalty",
                "max_new_tokens", "do_sample", "no_repeat_ngram_size"
            ]:
                if extra_col in cfg:
                    record_dict[f"gen_{extra_col}"] = cfg[extra_col]
                else:
                    record_dict[f"gen_{extra_col}"] = np.nan

            for meta_col in ["idFinal", "grupo", "motivo", "lex"]:
                if meta_col in row.index:
                    record_dict[meta_col] = row[meta_col]

            all_records.append(record_dict)

print(f"Corridas completadas: {len(all_records)}")

[1/1000] owner=hariel | model=llama3 | cfg=hariel_llama3_cfg_1 | ruleset=R0 | sample_id=2088
[2/1000] owner=hariel | model=llama3 | cfg=hariel_llama3_cfg_1 | ruleset=R0 | sample_id=2976
[3/1000] owner=hariel | model=llama3 | cfg=hariel_llama3_cfg_1 | ruleset=R0 | sample_id=3430
[4/1000] owner=hariel | model=llama3 | cfg=hariel_llama3_cfg_1 | ruleset=R0 | sample_id=3679
[5/1000] owner=hariel | model=llama3 | cfg=hariel_llama3_cfg_1 | ruleset=R0 | sample_id=3145
[6/1000] owner=hariel | model=llama3 | cfg=hariel_llama3_cfg_1 | ruleset=R0 | sample_id=507
[7/1000] owner=hariel | model=llama3 | cfg=hariel_llama3_cfg_1 | ruleset=R0 | sample_id=1756
[8/1000] owner=hariel | model=llama3 | cfg=hariel_llama3_cfg_1 | ruleset=R0 | sample_id=3093
[9/1000] owner=hariel | model=llama3 | cfg=hariel_llama3_cfg_1 | ruleset=R0 | sample_id=3192
[10/1000] owner=hariel | model=llama3 | cfg=hariel_llama3_cfg_1 | ruleset=R0 | sample_id=3525
[11/1000] owner=hariel | model=llama3 | cfg=hariel_llama3_cfg_1 | rule

In [11]:
raw_df = pd.DataFrame(all_records)

print("Shape raw_df:", raw_df.shape)
display(raw_df.head(3))
display(raw_df.columns.tolist())

Shape raw_df: (1000, 34)


,experiment_id,run_id,timestamp,dataset_name,fold_id,split_name,model_key,model_id,backend,prompt_type,...,gen_temperature,gen_top_p,gen_repetition_penalty,gen_max_new_tokens,gen_do_sample,gen_no_repeat_ngram_size,idFinal,grupo,motivo,lex
0,exp_ruleset_comparison_zero_20260502_235933,e418bdb2-05aa-4086-8925-c359e0ca81b3,2026-05-02T23:59:54.113466,muestra_20_ruleset_comparison_zero,None,ruleset_comparison_zero,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,zero-shot,...,0.2,0.85,1.05,256,NaN,NaN,1872_LibroBAC.pdf,A_cortos,"Muy corto, sustitución léxica clara.",1
1,exp_ruleset_comparison_zero_20260502_235933,b320b30c-96ac-45a2-9158-5af01b66fb3f,2026-05-02T23:59:57.347863,muestra_20_ruleset_comparison_zero,None,ruleset_comparison_zero,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,zero-shot,...,0.2,0.85,1.05,256,NaN,NaN,881_LibroNEFE_Sincopyright.pdf,A_cortos,Corto con dinero/cantidad.,1
2,exp_ruleset_comparison_zero_20260502_235933,8971c4ae-a2ac-4254-975c-3338111c4487,2026-05-03T00:00:00.685672,muestra_20_ruleset_comparison_zero,None,ruleset_comparison_zero,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,zero-shot,...,0.2,0.85,1.05,256,NaN,NaN,2692_LibroUide_Sincopyright.pdf,A_cortos,Frase conceptual simple de negocios.,1


['experiment_id',
 'run_id',
 'timestamp',
 'dataset_name',
 'fold_id',
 'split_name',
 'model_key',
 'model_id',
 'backend',
 'prompt_type',
 'ruleset',
 'few_shot_example_ids',
 'generation_config',
 'sample_id',
 'source_text',
 'reference_text',
 'generated_text',
 'prompt_text',
 'inference_seconds',
 'status',
 'error_message',
 'metrics',
 'owner',
 'config_label',
 'gen_temperature',
 'gen_top_p',
 'gen_repetition_penalty',
 'gen_max_new_tokens',
 'gen_do_sample',
 'gen_no_repeat_ngram_size',
 'idFinal',
 'grupo',
 'motivo',
 'lex']

In [12]:
if "status" in raw_df.columns:
    eval_input_df = raw_df[raw_df["status"] == "success"].copy()
else:
    eval_input_df = raw_df.copy()

print("Shape eval_input_df:", eval_input_df.shape)

evaluated_df = evaluate_dataframe(
    eval_input_df,
    source_col="source_text",
    pred_col="generated_text",
    ref_col="reference_text",
    compute_bertscore=True,
    compute_sbert=False,
)

print("Shape evaluated_df:", evaluated_df.shape)
display(evaluated_df.head(3))

Shape eval_input_df: (1000, 34)
Shape evaluated_df: (1000, 59)


,experiment_id,run_id,timestamp,dataset_name,fold_id,split_name,model_key,model_id,backend,prompt_type,...,additions_proportion,deletions_proportion,inflesz_pred,inflesz_src,inflesz_delta,rouge1_f,rouge2_f,rougeL_f,bertscore_f1,sbert_similarity
0,exp_ruleset_comparison_zero_20260502_235933,e418bdb2-05aa-4086-8925-c359e0ca81b3,2026-05-02T23:59:54.113466,muestra_20_ruleset_comparison_zero,None,ruleset_comparison_zero,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,zero-shot,...,0.222222,0.300000,52.468333,34.855000,17.613333,0.736842,0.705882,0.736842,0.912748,None
1,exp_ruleset_comparison_zero_20260502_235933,b320b30c-96ac-45a2-9158-5af01b66fb3f,2026-05-02T23:59:57.347863,muestra_20_ruleset_comparison_zero,None,ruleset_comparison_zero,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,zero-shot,...,0.285714,0.375000,112.735000,105.172500,7.562500,0.482759,0.222222,0.275862,0.791504,None
2,exp_ruleset_comparison_zero_20260502_235933,8971c4ae-a2ac-4254-975c-3338111c4487,2026-05-03T00:00:00.685672,muestra_20_ruleset_comparison_zero,None,ruleset_comparison_zero,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,zero-shot,...,0.307692,0.470588,69.235000,76.229118,-6.994118,0.640000,0.347826,0.320000,0.872574,None


In [21]:
group_df = evaluated_df.copy()

group_df["gen_do_sample"] = group_df["gen_do_sample"].astype("string").fillna("NA")
group_df["gen_no_repeat_ngram_size"] = group_df["gen_no_repeat_ngram_size"].astype("string").fillna("NA")

group_cols = [
    "owner",
    "model_key",
    "config_label",
    "ruleset",
    "gen_temperature",
    "gen_top_p",
    "gen_repetition_penalty",
    "gen_max_new_tokens",
    "gen_do_sample",
    "gen_no_repeat_ngram_size",
]

summary_by_ruleset = summarize_metrics(
    group_df,
    group_cols=group_cols,
)

print("Shape summary_by_ruleset:", summary_by_ruleset.shape)
display(summary_by_ruleset.head(10))

Shape summary_by_ruleset: (50, 28)


,owner,model_key,config_label,ruleset,gen_temperature,gen_top_p,gen_repetition_penalty,gen_max_new_tokens,gen_do_sample,gen_no_repeat_ngram_size,...,exact_copy,additions_proportion,deletions_proportion,rouge1_f,rouge2_f,rougeL_f,inflesz_pred,inflesz_src,inflesz_delta,bertscore_f1
0,hariel,llama3,hariel_llama3_cfg_1,R0,0.2,0.85,1.05,256,NA,NA,...,0.0,0.335613,0.509976,0.483206,0.288552,0.405020,73.239084,51.983615,21.255469,0.802757
1,hariel,llama3,hariel_llama3_cfg_1,R1,0.2,0.85,1.05,256,NA,NA,...,0.0,0.431450,0.502076,0.447493,0.252202,0.394901,68.925717,51.983615,16.942102,0.797565
2,hariel,llama3,hariel_llama3_cfg_1,R2,0.2,0.85,1.05,256,NA,NA,...,0.0,0.395192,0.557373,0.444645,0.230932,0.376377,75.044209,51.983615,23.060594,0.794482
3,hariel,llama3,hariel_llama3_cfg_1,R3,0.2,0.85,1.05,256,NA,NA,...,0.0,0.389705,0.542974,0.461568,0.242296,0.392203,75.814815,51.983615,23.831200,0.803366
4,hariel,llama3,hariel_llama3_cfg_1,R4,0.2,0.85,1.05,256,NA,NA,...,0.0,0.411557,0.562146,0.447494,0.253076,0.384787,69.091168,51.983615,17.107553,0.795653
5,hariel,llama3,hariel_llama3_cfg_2,R0,0.2,0.90,1.10,256,NA,NA,...,0.0,0.424026,0.540341,0.448140,0.248565,0.377270,73.694429,51.983615,21.710814,0.796184
6,hariel,llama3,hariel_llama3_cfg_2,R1,0.2,0.90,1.10,256,NA,NA,...,0.0,0.492650,0.608846,0.408136,0.203894,0.359976,73.680951,51.983615,21.697337,0.788665
7,hariel,llama3,hariel_llama3_cfg_2,R2,0.2,0.90,1.10,256,NA,NA,...,0.0,0.459629,0.577805,0.426677,0.234503,0.374492,74.012741,51.983615,22.029126,0.790770
8,hariel,llama3,hariel_llama3_cfg_2,R3,0.2,0.90,1.10,256,NA,NA,...,0.0,0.503851,0.609095,0.403995,0.193542,0.355399,74.281825,51.983615,22.298211,0.790559
9,hariel,llama3,hariel_llama3_cfg_2,R4,0.2,0.90,1.10,256,NA,NA,...,0.0,0.457797,0.575878,0.427764,0.234828,0.374822,72.689974,51.983615,20.706359,0.789294


In [22]:
summary_by_ruleset["leader_score"] = (
    0.6 * summary_by_ruleset["sari"] +
    0.4 * (summary_by_ruleset["bertscore_f1"] * 100.0)
)

display(
    summary_by_ruleset[
        [
            "owner",
            "model_key",
            "config_label",
            "ruleset",
            "sari",
            "bertscore_f1",
            "leader_score",
            "rougeL_f",
            "compression_ratio_eval",
            "exact_copy",
        ]
    ].sort_values(
        by=["leader_score", "sari", "bertscore_f1"],
        ascending=False
    ).head(20)
)

,owner,model_key,config_label,ruleset,sari,bertscore_f1,leader_score,rougeL_f,compression_ratio_eval,exact_copy
0,hariel,llama3,hariel_llama3_cfg_1,R0,41.984327,0.802757,57.300870,0.405020,0.746944,0.0
49,nancy,mistral,nancy_mistral_cfg_2,R4,41.904275,0.801483,57.201885,0.396466,0.902308,0.0
19,hariel,mistral,hariel_mistral_cfg_1,R4,41.313539,0.799005,56.748324,0.383889,1.010825,0.0
1,hariel,llama3,hariel_llama3_cfg_1,R1,40.927673,0.797565,56.459191,0.394901,0.889054,0.0
26,hariel,mistral,hariel_mistral_cfg_3,R1,40.852745,0.798477,56.450713,0.413970,1.047526,0.0
40,nancy,mistral,nancy_mistral_cfg_1,R0,40.336949,0.802013,56.282678,0.408220,0.972621,0.0
25,hariel,mistral,hariel_mistral_cfg_3,R0,40.552833,0.797208,56.220031,0.408561,0.972869,0.0
21,hariel,mistral,hariel_mistral_cfg_2,R1,40.565152,0.796415,56.195683,0.403377,1.110639,0.0
14,hariel,llama3,hariel_llama3_cfg_3,R4,40.659859,0.792474,56.094857,0.365241,0.843799,0.0
3,hariel,llama3,hariel_llama3_cfg_1,R3,39.830596,0.803366,56.032999,0.392203,0.769154,0.0


In [23]:
raw_path = OUTPUT_DIR / f"{EXPERIMENT_ID}_raw_results.csv"
eval_path = OUTPUT_DIR / f"{EXPERIMENT_ID}_evaluated.csv"
summary_path = OUTPUT_DIR / f"{EXPERIMENT_ID}_summary_by_ruleset.csv"

raw_df.to_csv(raw_path, index=False, encoding="utf-8-sig")
evaluated_df.to_csv(eval_path, index=False, encoding="utf-8-sig")
summary_by_ruleset.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("Raw guardado en:", raw_path)
print("Evaluated guardado en:", eval_path)
print("Summary by ruleset guardado en:", summary_path)

Raw guardado en: /home/harielpadillasanchez/Documentos/TT/TT2/outputs/ruleset_comparison_zero/exp_ruleset_comparison_zero_20260502_235933_raw_results.csv
Evaluated guardado en: /home/harielpadillasanchez/Documentos/TT/TT2/outputs/ruleset_comparison_zero/exp_ruleset_comparison_zero_20260502_235933_evaluated.csv
Summary by ruleset guardado en: /home/harielpadillasanchez/Documentos/TT/TT2/outputs/ruleset_comparison_zero/exp_ruleset_comparison_zero_20260502_235933_summary_by_ruleset.csv


In [24]:
metadata = {
    "experiment_id": EXPERIMENT_ID,
    "prompt_type": PROMPT_TYPE,
    "n_configs": len(FINALIST_CONFIGS),
    "n_rulesets": len(ACTIVE_RULESETS),
    "n_rows_sample": len(df_refine),
    "expected_total_runs": total_runs,
    "raw_path": str(raw_path),
    "eval_path": str(eval_path),
    "summary_path": str(summary_path),
}

meta_path = OUTPUT_DIR / f"{EXPERIMENT_ID}_metadata.json"
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("Metadata guardada en:", meta_path)

Metadata guardada en: /home/harielpadillasanchez/Documentos/TT/TT2/outputs/ruleset_comparison_zero/exp_ruleset_comparison_zero_20260502_235933_metadata.json


In [25]:
ranking_zero = summary_by_ruleset.sort_values(
    by=["leader_score", "sari", "bertscore_f1"],
    ascending=False
).reset_index(drop=True)

display(
    ranking_zero[
        [
            "owner",
            "model_key",
            "config_label",
            "ruleset",
            "gen_temperature",
            "gen_top_p",
            "gen_repetition_penalty",
            "gen_max_new_tokens",
            "gen_do_sample",
            "gen_no_repeat_ngram_size",
            "sari",
            "bertscore_f1",
            "leader_score",
            "rougeL_f",
            "compression_ratio_eval",
            "exact_copy",
        ]
    ].head(20)
)

ranking_zero_path = OUTPUT_DIR / f"{EXPERIMENT_ID}_ranking_zero_global.csv"
ranking_zero.to_csv(ranking_zero_path, index=False, encoding="utf-8-sig")

print("Ranking zero global guardado en:", ranking_zero_path)

,owner,model_key,config_label,ruleset,gen_temperature,gen_top_p,gen_repetition_penalty,gen_max_new_tokens,gen_do_sample,gen_no_repeat_ngram_size,sari,bertscore_f1,leader_score,rougeL_f,compression_ratio_eval,exact_copy
0,hariel,llama3,hariel_llama3_cfg_1,R0,0.2,0.85,1.05,256,NA,NA,41.984327,0.802757,57.300870,0.405020,0.746944,0.0
1,nancy,mistral,nancy_mistral_cfg_2,R4,0.3,0.90,1.15,400,True,4.0,41.904275,0.801483,57.201885,0.396466,0.902308,0.0
2,hariel,mistral,hariel_mistral_cfg_1,R4,0.3,0.90,1.10,256,NA,NA,41.313539,0.799005,56.748324,0.383889,1.010825,0.0
3,hariel,llama3,hariel_llama3_cfg_1,R1,0.2,0.85,1.05,256,NA,NA,40.927673,0.797565,56.459191,0.394901,0.889054,0.0
4,hariel,mistral,hariel_mistral_cfg_3,R1,0.2,0.85,1.10,256,NA,NA,40.852745,0.798477,56.450713,0.413970,1.047526,0.0
5,nancy,mistral,nancy_mistral_cfg_1,R0,0.7,0.90,1.10,512,True,4.0,40.336949,0.802013,56.282678,0.408220,0.972621,0.0
6,hariel,mistral,hariel_mistral_cfg_3,R0,0.2,0.85,1.10,256,NA,NA,40.552833,0.797208,56.220031,0.408561,0.972869,0.0
7,hariel,mistral,hariel_mistral_cfg_2,R1,0.1,0.85,1.10,256,NA,NA,40.565152,0.796415,56.195683,0.403377,1.110639,0.0
8,hariel,llama3,hariel_llama3_cfg_3,R4,0.3,0.90,1.15,256,NA,NA,40.659859,0.792474,56.094857,0.365241,0.843799,0.0
9,hariel,llama3,hariel_llama3_cfg_1,R3,0.2,0.85,1.05,256,NA,NA,39.830596,0.803366,56.032999,0.392203,0.769154,0.0


Ranking zero global guardado en: /home/harielpadillasanchez/Documentos/TT/TT2/outputs/ruleset_comparison_zero/exp_ruleset_comparison_zero_20260502_235933_ranking_zero_global.csv


In [26]:
top5_by_model_zero = (
    ranking_zero
    .sort_values(
        by=["model_key", "leader_score", "sari", "bertscore_f1"],
        ascending=[True, False, False, False]
    )
    .groupby("model_key", as_index=False, group_keys=False)
    .head(5)
    .reset_index(drop=True)
)

display(
    top5_by_model_zero[
        [
            "owner",
            "model_key",
            "config_label",
            "ruleset",
            "gen_temperature",
            "gen_top_p",
            "gen_repetition_penalty",
            "gen_max_new_tokens",
            "gen_do_sample",
            "gen_no_repeat_ngram_size",
            "sari",
            "bertscore_f1",
            "leader_score",
        ]
    ]
)

top5_by_model_zero_path = OUTPUT_DIR / f"{EXPERIMENT_ID}_top5_by_model_zero.csv"
top5_by_model_zero.to_csv(top5_by_model_zero_path, index=False, encoding="utf-8-sig")

print("Top5 by model zero guardado en:", top5_by_model_zero_path)

,owner,model_key,config_label,ruleset,gen_temperature,gen_top_p,gen_repetition_penalty,gen_max_new_tokens,gen_do_sample,gen_no_repeat_ngram_size,sari,bertscore_f1,leader_score
0,hariel,llama3,hariel_llama3_cfg_1,R0,0.2,0.85,1.05,256,NA,NA,41.984327,0.802757,57.300870
1,hariel,llama3,hariel_llama3_cfg_1,R1,0.2,0.85,1.05,256,NA,NA,40.927673,0.797565,56.459191
2,hariel,llama3,hariel_llama3_cfg_3,R4,0.3,0.90,1.15,256,NA,NA,40.659859,0.792474,56.094857
3,hariel,llama3,hariel_llama3_cfg_1,R3,0.2,0.85,1.05,256,NA,NA,39.830596,0.803366,56.032999
4,hariel,llama3,hariel_llama3_cfg_1,R4,0.2,0.85,1.05,256,NA,NA,40.174015,0.795653,55.930532
5,nancy,mistral,nancy_mistral_cfg_2,R4,0.3,0.90,1.15,400,True,4.0,41.904275,0.801483,57.201885
6,hariel,mistral,hariel_mistral_cfg_1,R4,0.3,0.90,1.10,256,NA,NA,41.313539,0.799005,56.748324
7,hariel,mistral,hariel_mistral_cfg_3,R1,0.2,0.85,1.10,256,NA,NA,40.852745,0.798477,56.450713
8,nancy,mistral,nancy_mistral_cfg_1,R0,0.7,0.90,1.10,512,True,4.0,40.336949,0.802013,56.282678
9,hariel,mistral,hariel_mistral_cfg_3,R0,0.2,0.85,1.10,256,NA,NA,40.552833,0.797208,56.220031


Top5 by model zero guardado en: /home/harielpadillasanchez/Documentos/TT/TT2/outputs/ruleset_comparison_zero/exp_ruleset_comparison_zero_20260502_235933_top5_by_model_zero.csv


In [27]:
top5_by_owner_zero = (
    ranking_zero
    .sort_values(
        by=["owner", "leader_score", "sari", "bertscore_f1"],
        ascending=[True, False, False, False]
    )
    .groupby("owner", as_index=False, group_keys=False)
    .head(5)
    .reset_index(drop=True)
)

display(
    top5_by_owner_zero[
        [
            "owner",
            "model_key",
            "config_label",
            "ruleset",
            "gen_temperature",
            "gen_top_p",
            "gen_repetition_penalty",
            "gen_max_new_tokens",
            "gen_do_sample",
            "gen_no_repeat_ngram_size",
            "sari",
            "bertscore_f1",
            "leader_score",
        ]
    ]
)

top5_by_owner_zero_path = OUTPUT_DIR / f"{EXPERIMENT_ID}_top5_by_owner_zero.csv"
top5_by_owner_zero.to_csv(top5_by_owner_zero_path, index=False, encoding="utf-8-sig")

print("Top5 by owner zero guardado en:", top5_by_owner_zero_path)

,owner,model_key,config_label,ruleset,gen_temperature,gen_top_p,gen_repetition_penalty,gen_max_new_tokens,gen_do_sample,gen_no_repeat_ngram_size,sari,bertscore_f1,leader_score
0,hariel,llama3,hariel_llama3_cfg_1,R0,0.2,0.85,1.05,256,NA,NA,41.984327,0.802757,57.300870
1,hariel,mistral,hariel_mistral_cfg_1,R4,0.3,0.90,1.10,256,NA,NA,41.313539,0.799005,56.748324
2,hariel,llama3,hariel_llama3_cfg_1,R1,0.2,0.85,1.05,256,NA,NA,40.927673,0.797565,56.459191
3,hariel,mistral,hariel_mistral_cfg_3,R1,0.2,0.85,1.10,256,NA,NA,40.852745,0.798477,56.450713
4,hariel,mistral,hariel_mistral_cfg_3,R0,0.2,0.85,1.10,256,NA,NA,40.552833,0.797208,56.220031
5,nancy,mistral,nancy_mistral_cfg_2,R4,0.3,0.90,1.15,400,True,4.0,41.904275,0.801483,57.201885
6,nancy,mistral,nancy_mistral_cfg_1,R0,0.7,0.90,1.10,512,True,4.0,40.336949,0.802013,56.282678
7,nancy,llama3,nancy_llama3_cfg_2,R2,0.7,0.90,1.10,512,True,4.0,40.348848,0.788883,55.764627
8,nancy,mistral,nancy_mistral_cfg_2,R0,0.3,0.90,1.15,400,True,4.0,39.777972,0.796144,55.712558
9,nancy,mistral,nancy_mistral_cfg_1,R3,0.7,0.90,1.10,512,True,4.0,40.738447,0.776105,55.487282


Top5 by owner zero guardado en: /home/harielpadillasanchez/Documentos/TT/TT2/outputs/ruleset_comparison_zero/exp_ruleset_comparison_zero_20260502_235933_top5_by_owner_zero.csv


In [28]:
print("Contenido generado en OUTPUT_DIR:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print("-", p.name)

Contenido generado en OUTPUT_DIR:
- exp_ruleset_comparison_zero_20260502_235933_evaluated.csv
- exp_ruleset_comparison_zero_20260502_235933_metadata.json
- exp_ruleset_comparison_zero_20260502_235933_ranking_zero_global.csv
- exp_ruleset_comparison_zero_20260502_235933_raw_results.csv
- exp_ruleset_comparison_zero_20260502_235933_summary_by_ruleset.csv
- exp_ruleset_comparison_zero_20260502_235933_top5_by_model_zero.csv
- exp_ruleset_comparison_zero_20260502_235933_top5_by_owner_zero.csv
